# OA Potential Fit with Poisson-Weighted Alpha

Builds a Fourier-Morse potential for the PA/OA interaction where each
per-orientation Morse ``alpha`` is fitted with a **Poisson weight** of
parameter $\lambda = 0.5$ centred at that orientation's $r_e$ (see
``07_weight_functions.ipynb``). The alpha weights suppress the steep
repulsive (small-$r$) side while retaining the large-$r$ tail.

The harmonic ceilings are kept at OA $(20, 1)$ to match the equal-weight
baseline used by ``plot_oa_error.py``, so the two error plots are directly
comparable. The fitted energy surface is exported to a CSV that can be fed
to ``plot_oa_error.py --fit-input``.

In [ ]:
from pathlib import Path

from chimorse.config import load_molecule_info
from chimorse.datasets import ensure_reference_data
from chimorse.dataio import load_data
from chimorse.fitting import generate_fourier_morse_data, make_weight_func

In [ ]:
molecule_name = 'PA'
interaction   = 'OA'
zero_zeta     = True
alpha_fit     = True

# Harmonic ceilings — OA (20, 1) to match the equal-weight baseline.
harmonic_ceils = {'EP': (8, 1), 'EA': (8, 1), 'OP': (20, 1), 'OA': (20, 1)}

# Poisson weight for the alpha fit, lambda = 0.5 (mode at each r_e).
weight_func = make_weight_func('poisson', lam=0.5)

data_root = Path('../data')
data_dir = ensure_reference_data(molecule_name, data_root=data_root)
molecule = load_molecule_info(molecule_name, metadata_path=data_dir / 'metadata.json')
df = load_data(molecule, interaction, zero_zeta=zero_zeta)
print(f'{molecule.name} {interaction}: {len(df)} rows')

In [ ]:
df_model = generate_fourier_morse_data(
    df, molecule, interaction, harmonic_ceils,
    alpha_fit=alpha_fit, weight_func=weight_func,
    print_errors=True, near_eq_delta_r=.5,
)
print('model rows:', len(df_model))

In [ ]:
out_csv = data_dir / f'df_model_{interaction}_w_poisson_lam0.5.csv'
df_model.to_csv(out_csv, index=False)
print('saved fit:', out_csv)

### Next step

Run ``plot_oa_error.py --fit-input <out_csv> --tag w_poisson_lam0.5``
to generate the corresponding error plots.